# Elasticity Examples

This is a jupyter notebook. The "real" notebook can be found [here](https://github.com/polyfem/polyfem.github.io/blob/docs/docs/elasticity.ipynb).

Some imports for plotting, algebra, and PolyFEM

In [ ]:
import meshplot as mp
import numpy as np
import polyfempy as pf
import utils

## Plate hole

This is the python version of the plate with hole example explained [here](https://polyfem.github.io/tutorial/#boundary-conditions).

Set the mesh path

In [ ]:
mesh_path = "plane_hole.obj"

create settings:

- Pick linear $P_1$ elements (if the mesh would be a quad it would be $Q_1$)
- We are use a linear material model

In [ ]:
settings = pf.Settings(
    discr_order=1,
    pde=pf.PDEs.LinearElasticity
)

and choose Young's modulus and poisson ratio

In [ ]:
settings.set_material_params("E", 210000)
settings.set_material_params("nu", 0.3)

Next we setup the problem

In [ ]:
problem = pf.Problem()

sideset 1 has symetric boundary in $x$

In [ ]:
problem.set_x_symmetric(1)

sideset 4 has symmetric boundary in $y$

In [ ]:
problem.set_y_symmetric(4)

sideset 3 has a force of [100, 0] applied

In [ ]:
problem.set_force(3, [100, 0])

fianally set the problem

In [ ]:
settings.problem = problem

Solve!
Note: we normalize the mesh to be in $[0,1]^2$

In [ ]:
solver = pf.Solver()

solver.settings(settings)
solver.load_mesh_from_path(mesh_path, normalize_mesh=True)

solver.solve()

Get the solution

In [ ]:
pts, tets, disp = solver.get_sampled_solution()

diplace the mesh

In [ ]:
vertices = pts + disp

and get the stresses

In [ ]:
mises, _ = solver.get_sampled_mises_avg()

finally plot with the above code

In [ ]:
mp.plot(vertices, tets, mises)

Note that we used `get_sampled_mises_avg` to get the Von Mises stresses because they depend on the Jacobian of the displacement which, becase we use $P_1$ elements, is piece-wise constant. To avoid that effect in `get_sampled_mises_avg` the mises are averaged per vertex weighted by the area of the triangles. If you want the "real" mises just call

In [ ]:
mises = solver.get_sampled_mises()
mp.plot(vertices, tets, mises)

## Plate hole with High-Order Mesh

This is the same example as before, but we use `wildmeshing` to create a curved mesh.

In [ ]:
import wildmeshing as wm
mesh_path = "plane_hole.svg"

v, f, nodes, F_nodes = wm.triangulate_svg(mesh_path, cut_outside=True)

Now we proceed as before

In [ ]:
#create settings
settings = pf.Settings(
    discr_order=1, #pick linear P_1 elements, even if the geometry is P_3
    pde=pf.PDEs.LinearElasticity #Linear elasticity
)

#Material parameters
settings.set_material_params("E", 210000)
settings.set_material_params("nu", 0.3)

Next we setup the problem as before

In [ ]:
problem = pf.Problem()

#sideset 1 is symmetric in x
problem.set_x_symmetric(1)

#sideset 4 is symmetric in y
problem.set_y_symmetric(4)

#sideset 3 has a force of [100, 0] applied
problem.set_force(3, [100, 0])

fianally set the problem

In [ ]:
settings.problem = problem

Create the solver and load the high-order mesh, the only difference with respect to before

In [ ]:
solver = pf.Solver()

solver.settings(settings)
solver.set_high_order_mesh(v, f, nodes, F_nodes, normalize_mesh=True)

And finally, solve!

In [ ]:
solver.solve()

Get and plot the solution, same as before

In [ ]:
pts, tets, disp = solver.get_sampled_solution()

#diplace the mesh
vertices = pts + disp

#get the stresses
mises, _ = solver.get_sampled_mises_avg()

#plot
mp.plot(vertices, tets, mises)

## Torsion

Non-linear example. We want to torque a 3D bar around the $z$ direction.

The example is really similar as the one just above.

Sets the mesh, create a solver, and load the mesh.

In this case the mesh has already the correct size. We also choose a coarse visualization mesh

In [ ]:
mesh_path = "square_beam_h.HYBRID"
solver = pf.Solver()
solver.load_mesh_from_path(mesh_path, normalize_mesh=False, vismesh_rel_area=0.001)

We want to use the default sideset marking, top of the mesh is 5 and bottom is 6.

Let's verify. We first extract the sidesets: `p` are some point, `t` triangles, and `s` the sidesets from 1 to 6

In [ ]:
p, t, s = solver.get_boundary_sidesets()

Now we can plot it

In [ ]:
tmp = np.zeros_like(s)
tmp[s==5] = 1
tmp[s==6] = 1

mp.plot(p, t, tmp)

Now we create the settings, as before.

**Note**: It is an hex-mesh so we are using $Q_1$.

Differently from before we want a non-linear material model: NeoHookean.

In [ ]:
settings = pf.Settings(
    discr_order=1,
    pde=pf.PDEs.NonLinearElasticity
)

Choose Young's modulus and Poisson's ratio, as before

In [ ]:
settings.set_material_params("E", 200)
settings.set_material_params("nu", 0.35)

Now we setup problem with fixed sideset (5), rotating sideset (6), ahlf a tour along the $z$-axis.

In [ ]:
problem = pf.Torsion(
    fixed_boundary=5, #sideset 5 is fixed
    turning_boundary=6, #sideset 6 rotates
    
    n_turns = 0.5, #by half a tour
    
    axis_coordiante=2, #around the z-axis, 2
)

and set the problem and solve

In [ ]:
settings.problem = problem

#solving!
solver.settings(settings)

solver.solve()

takes approx 1 min because it is a complicated non-linear problem!

Get solution and stesses as before

Since we want to show only the surface there is no need of getting the whole volume, so we set `boundary_only` to `True`

In [ ]:
pts, tets, disp = solver.get_sampled_solution(boundary_only=True)
vertices = pts + disp
mises, _ = solver.get_sampled_mises_avg(boundary_only=True)

and plot the 3d result!

In [ ]:
mp.plot(vertices, tets, mises, shading={"flat":True}, return_plot=True)

## Time dependent simulation

Create the mesh using the utility function

In [ ]:
pts, faces = utils.create_quad_mesh(50)

create settings, pick linear $Q_1$ elements, and a linear material model.

In this case we want to run a simulation with $t\in[0, 10]$ and have 50 time steps.

In [ ]:
settings = pf.Settings(
    discr_order=1,
    pde=pf.PDEs.LinearElasticity,
    
    tend=10,
    time_steps=50
)

Choose Young's modulus and poisson ratio

In [ ]:
settings.set_material_params("E", 1)
settings.set_material_params("nu", 0.3)

Next we setup the problem, this doesnt have any parameters. It will...

In [ ]:
problem = pf.Gravity()

we set the problem

In [ ]:
settings.problem = problem

We create the solver and set the settings

In [ ]:
solver = pf.Solver()
solver.settings(settings)

This time we are using `pts` and `faces` instead of loading from the disk (For efficienty in the browser we select a coarse vis mesh)

In [ ]:
solver.set_mesh(pts, faces, vismesh_rel_area=0.001)

Solve!

In [ ]:
solver.solve()

Get the solution and the mises

In [ ]:
pts = solver.get_sampled_points_frames()
tris = solver.get_sampled_connectivity_frames()
disp = solver.get_sampled_solution_frames()
mises = solver.get_sampled_mises_avg_frames()

In [ ]:
def plot(frame, p=None):
    return mp.plot(pts[frame]+disp[frame], tris[frame], mises[frame], return_plot=True, plot=p)

Before the animation, let's plot the solution some frames

In [ ]:
plot(0)

In [ ]:
plot(5)

In [ ]:
plot(11)

Now we are ready to do the animation

In [ ]:
p = plot(0)

@mp.interact(frame=(0, len(mises)-1))
def step(frame=0):
    plot(frame, p)